# Tenacious-Bench — ORPO Inference on Held-Out Split

Generates two prediction files needed by `run_ablation.py`:
- `prompt_only_predictions.jsonl` — bare `unsloth/Qwen3-0.6B`, no adapter
- `trained_predictions.jsonl` — same base + ORPO LoRA adapter

**Before running:** make sure the Colab runtime is set to **GPU (T4)**.

**Files to upload when prompted (Cell 3):**
- `hand_authored_tasks.jsonl` — from `tenacious_bench_v0.1/held_out/`
- `adapter_config.json` — from `models/orpo_qwen3_0_6b_lora_adapter/`
- `adapter_model.safetensors` — from `models/orpo_qwen3_0_6b_lora_adapter/`

**Download at the end (Cell 7):** both prediction JSONL files, then copy them into `reports/` on your local machine.

In [ ]:
# Cell 1 — install dependencies
!pip install -q transformers peft accelerate safetensors

In [ ]:
# Cell 2 — verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected — inference will be slow on CPU')

In [ ]:
# Cell 3 — upload required files
#
# Upload these three files when the file picker opens:
#   hand_authored_tasks.jsonl       (from tenacious_bench_v0.1/held_out/)
#   adapter_config.json             (from models/orpo_qwen3_0_6b_lora_adapter/)
#   adapter_model.safetensors       (from models/orpo_qwen3_0_6b_lora_adapter/)

from google.colab import files
import os, shutil

uploaded = files.upload()

# Move adapter files into a dedicated folder
os.makedirs('/content/adapter', exist_ok=True)
for fname in ('adapter_config.json', 'adapter_model.safetensors'):
    if fname in uploaded:
        with open(f'/content/adapter/{fname}', 'wb') as f:
            f.write(uploaded[fname])
        print(f'Saved adapter/{fname}')

# Save held-out tasks
if 'hand_authored_tasks.jsonl' in uploaded:
    with open('/content/hand_authored_tasks.jsonl', 'wb') as f:
        f.write(uploaded['hand_authored_tasks.jsonl'])
    print('Saved hand_authored_tasks.jsonl')

# Count tasks
import json
tasks = [json.loads(l) for l in open('/content/hand_authored_tasks.jsonl') if l.strip()]
print(f'Loaded {len(tasks)} held-out tasks')

In [ ]:
# Cell 4 — prompt builder and output parser (mirrors ORPO training format)
import re

_THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL | re.IGNORECASE)
_SUBJECT_RE = re.compile(r'(?i)^subject:\s*(.+)', re.MULTILINE)
_BODY_RE = re.compile(r'(?i)^body:\s*([\s\S]+)', re.MULTILINE)


def _signals_text(brief):
    lines = []
    for sig in brief.get('signals', []):
        if isinstance(sig, dict):
            stype = sig.get('signal_type', 'signal')
            conf  = sig.get('confidence', 'medium')
            text  = sig.get('evidence') or sig.get('signal_type', '')
        else:
            stype, conf, text = 'signal', 'medium', str(sig)
        lines.append(f'- {stype} ({conf}): {text}')
    return '\n'.join(lines) if lines else '- no signals provided'


def build_prompt(task):
    inp      = task.get('input', {})
    prospect = inp.get('prospect', {})
    brief    = inp.get('hiring_signal_brief', {})
    prior    = inp.get('prior_thread', {})
    rubric   = task.get('rubric', {})
    sc       = task.get('scoring_config', {})
    gt       = task.get('ground_truth', {})

    channel        = task.get('channel', 'email')
    message_kind   = task.get('message_kind', 'cold_outreach')
    company_name   = prospect.get('company_name', 'Unknown')
    contact_role   = prospect.get('contact_role', '')
    company_stage  = prospect.get('company_stage', '')
    failure_dim    = task.get('failure_dimension', '')
    judge_dims     = ', '.join(sc.get('judge_dimensions', []))
    prior_summary  = prior.get('summary', 'No prior contact.')
    expected_beh   = gt.get('expected_behavior', '')
    required       = ', '.join(rubric.get('expected_terms', []))
    forbidden      = ', '.join(rubric.get('forbidden_terms', []))
    banned         = ', '.join(rubric.get('banned_phrases', []))

    return (
        f'Write a {channel} {message_kind} for {contact_role} at {company_name} ({company_stage}).\n'
        f'Failure dimension focus: {failure_dim}.\n'
        f'Judge dimensions: {judge_dims}.\n'
        f'Prior thread summary: {prior_summary}\n'
        f'Signals:\n{_signals_text(brief)}\n'
        f'Expected behavior: {expected_beh}\n'
        f'Required terms: {required}\n'
        f'Forbidden terms: {forbidden}\n'
        f'Banned phrases: {banned}\n'
        'Keep the response grounded, professional, and limited to one buyer-facing next step.'
    )


def parse_output(raw):
    text = _THINK_RE.sub('', raw).strip()
    subject_m = _SUBJECT_RE.search(text)
    body_m    = _BODY_RE.search(text)
    subject   = subject_m.group(1).strip() if subject_m else ''
    body      = body_m.group(1).strip()    if body_m    else text
    body = body.split('<|im_end|>')[0].split('<|endoftext|>')[0].strip()
    return {'subject': subject, 'body': body}


print('Helpers defined.')

In [ ]:
# Cell 5 — load base model and tokenizer (shared by both inference passes)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL = 'unsloth/Qwen3-0.6B'
dtype      = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f'Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading base model (dtype={dtype}) ...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map='auto',
    trust_remote_code=True,
)
base_model.eval()
print('Base model ready.')

In [ ]:
# Cell 6 — inference function and main loop
import time

MAX_NEW_TOKENS = 300

def run_inference(model, prompt):
    messages   = [{'role': 'user', 'content': prompt}]
    input_ids  = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt'
    )
    device    = next(model.parameters()).device
    input_ids = input_ids.to(device)
    t0 = time.monotonic()
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency_ms = (time.monotonic() - t0) * 1000
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return raw, round(latency_ms, 1)


def generate_predictions(model, tasks, output_path):
    written = skipped = 0
    with open(output_path, 'w') as fh:
        for i, task in enumerate(tasks, 1):
            task_id = task.get('task_id', f'task_{i}')
            try:
                prompt = build_prompt(task)
                raw, latency_ms = run_inference(model, prompt)
                candidate = parse_output(raw)
            except Exception as exc:
                print(f'  SKIP {task_id}: {exc}')
                skipped += 1
                continue
            row = {
                'task_id': task_id,
                'candidate_output': candidate,
                'latency_ms': latency_ms,
                'input_tokens': 0,
                'output_tokens': 0,
                'usd_cost': 0.0,
            }
            fh.write(json.dumps(row) + '\n')
            written += 1
            if i % 10 == 0:
                print(f'  {i}/{len(tasks)} done')
    print(f'Done - wrote {written}, skipped {skipped} -> {output_path}')


# --- Pass 1: prompt-only (bare base model) ---
print('=== Pass 1: prompt-only ===')
generate_predictions(base_model, tasks, '/content/prompt_only_predictions.jsonl')

In [ ]:
# Cell 7 — load LoRA adapter and run trained pass
# Upgrade torchao first to fix the version incompatibility peft requires
import subprocess, sys

print('Upgrading torchao ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torchao'], check=True)

# Clear any stale peft/torchao modules so the fresh versions are picked up
for mod in list(sys.modules.keys()):
    if 'peft' in mod or 'torchao' in mod:
        del sys.modules[mod]

from peft import PeftModel

print('Attaching LoRA adapter ...')
trained_model = PeftModel.from_pretrained(base_model, '/content/adapter')
trained_model = trained_model.merge_and_unload()
trained_model.eval()
print('Adapter merged. Running trained pass ...')

print('=== Pass 2: trained (ORPO adapter) ===')
generate_predictions(trained_model, tasks, '/content/trained_predictions.jsonl')

In [ ]:
# Cell 8 — spot-check: print first row of each file side by side
def first_row(path):
    with open(path) as f:
        return json.loads(f.readline())

po = first_row('/content/prompt_only_predictions.jsonl')
tr = first_row('/content/trained_predictions.jsonl')

print('--- PROMPT-ONLY ---')
print('task_id:', po['task_id'])
print('subject:', po['candidate_output']['subject'])
print('body:', po['candidate_output']['body'][:200])

print()
print('--- TRAINED ---')
print('task_id:', tr['task_id'])
print('subject:', tr['candidate_output']['subject'])
print('body:', tr['candidate_output']['body'][:200])

In [ ]:
# Cell 9 — download both files to your local machine
# Copy them into reports/ in the Sales Eval Bench repo, then run:
#
#   python src/ablations/run_ablation.py \
#     --comparison all \
#     --baseline-predictions reports/baseline_predictions.jsonl \
#     --trained-predictions reports/trained_predictions.jsonl \
#     --prompt-only-predictions reports/prompt_only_predictions.jsonl \
#     --output reports/ablation_summary.json

from google.colab import files
files.download('/content/prompt_only_predictions.jsonl')
files.download('/content/trained_predictions.jsonl')